# Data Preparation and Ground Truth Construction

This notebook implements the preprocessing pipeline for the Kvasir-Capsule dataset. 
The workflow consists of:-
(i) Extraction of annotated regions from metadata.
(ii) frame-level processing with bounding box overlays.
(iii) construction of structured ground truth for downstream evaluation.

## Setup

In [ ]:
import json
from pathlib import Path
import cv2
import numpy as np
from tqdm import tqdm

In [ ]:
BASE_DIR = Path.cwd()
JSON_PATH = BASE_DIR / "metadata.json"
OUTPUT_DIR = BASE_DIR / "output"
PADDING = 30

TARGET_CLASSES = {'Ulcer', 'Erosion', 'Blood - fresh', 'Polyp'}

## Bounding Box Extraction

In [ ]:
def shape_to_bbox(shape):
    xs = [p['x'] for p in shape]
    ys = [p['y'] for p in shape]
    return int(min(xs)), int(min(ys)), int(max(xs)), int(max(ys))

In [ ]:
def draw_bbox(frame, x1, y1, x2, y2):
    out = frame.copy()
    cv2.rectangle(out, (x1, y1), (x2, y2), (0,255,0), 2)
    return out

## Metadata Loading

In [ ]:
with open(JSON_PATH, encoding='utf-8') as f:
    data = json.load(f)

## Frame Processing

In [ ]:
video_files = sorted(BASE_DIR.glob('*.mp4'))

for video_path in video_files:
    vid_id = video_path.stem
    if vid_id not in data:
        continue

    findings = data[vid_id]['findings']
    cap = cv2.VideoCapture(str(video_path))

    for fid, finding in findings.items():
        subtype = finding['metadata'].get('pillcam_subtype')
        if subtype not in TARGET_CLASSES:
            continue

        frames = finding['frames']
        if not frames:
            continue

        frame_nums = sorted(int(fn) for fn in frames.keys())
        clip_start = max(0, frame_nums[0] - PADDING)
        clip_end   = min(int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) - 1, frame_nums[-1] + PADDING)

        cap.set(cv2.CAP_PROP_POS_FRAMES, clip_start)

        for fn in range(clip_start, clip_end + 1):
            ret, frame = cap.read()
            if not ret:
                break

            if fn in [int(k) for k in frames.keys()]:
                x1, y1, x2, y2 = shape_to_bbox(frames[str(fn)]['shape'])
                frame = draw_bbox(frame, x1, y1, x2, y2)

        cap.release()

## Ground Truth Construction

In [ ]:
records = []

for video_path in video_files:
    vid_id = video_path.stem
    if vid_id not in data:
        continue

    findings = data[vid_id]['findings']

    for fid, finding in findings.items():
        subtype = finding['metadata'].get('pillcam_subtype')
        if subtype not in TARGET_CLASSES:
            continue

        frames = finding['frames']
        if not frames:
            continue

        frame_nums = sorted(int(x) for x in frames.keys())

        bboxes = []
        for fn_str, ann in frames.items():
            x1, y1, x2, y2 = shape_to_bbox(ann['shape'])
            bboxes.append({
                'frame': int(fn_str),
                'x1': x1, 'y1': y1,
                'x2': x2, 'y2': y2
            })

        records.append({
            'video_id': vid_id,
            'finding_id': fid,
            'class': subtype,
            'frame_start': frame_nums[0],
            'frame_end': frame_nums[-1],
            'bboxes': bboxes
        })

## Output

In [ ]:
with open("ground_truth.json", "w") as f:
    json.dump(records, f, indent=2)

len(records)

## Visualization

In [ ]:
sample_video = video_files[0]
cap = cv2.VideoCapture(str(sample_video))
ret, frame = cap.read()
cap.release()

bbox = [100,100,300,300]
frame_box = draw_bbox(frame, *bbox)

import matplotlib.pyplot as plt
plt.imshow(cv2.cvtColor(frame_box, cv2.COLOR_BGR2RGB))
plt.axis('off')